# Sepsis Early Prediction — PhysioNet/CinC Challenge 2019
**Project 2 of the medical-AI portfolio.** Predict, for each hour in the ICU, whether a patient is heading toward sepsis, from hourly vitals and labs.

This notebook runs end to end: **download → EDA → preprocessing → gradient-boosted baseline → LSTM → comparison.**

**Two models, same task (per-hour prediction):**
- **Baseline** — gradient-boosted trees on each patient-hour (no memory of earlier hours). The number every sequence model must beat.
- **LSTM** — reads the patient's hourly *sequence*, so it can use trend and history.

**How to run:** `Runtime → Change runtime type → GPU`, then `Runtime → Run all`. Expect ~10–15 min (download ~1 min, loading ~3–5 min, LSTM a few min on GPU).

**The central challenge is imbalance** — sepsis-positive hours are only ~1–2% of all hours — so we watch **AUPRC** (precision–recall) more than AUROC, and up-weight the rare positive class during training.

## Step 1 — Download the data
~40,000 ICU patients (hospital A + B) from PhysioNet's official AWS mirror, open access. Each `.psv` is one patient: pipe-separated, **one row per hour**, 40 features + `SepsisLabel`. This is the slowest cell (~40k small files) — give it a minute or two.

In [ ]:
# Download PhysioNet 2019 from the official AWS mirror (open access, no account)
!pip install -q awscli
!mkdir -p /content/sepsis
!aws s3 sync --no-sign-request s3://physionet-open/challenge-2019/1.0.0/training/ /content/sepsis/ --quiet

import glob
files = sorted(glob.glob("/content/sepsis/**/*.psv", recursive=True))
print("patient files:", len(files), "(expect ~40,000)")
assert files, "download produced no files — re-run this cell"


## Step 2 — Setup
Imports, config, and seeds (for reproducibility). If `device` prints `cpu`, switch to a GPU runtime for a much faster LSTM.

In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

# ---------- config ----------
SEED         = 42
MAX_LEN      = 72      # cap each patient to their last MAX_LEN hours (bounds memory)
HIDDEN       = 64
BATCH        = 64
EPOCHS       = 5
MAX_PATIENTS = None    # None = use all ~40k; set e.g. 8000 for a quick CPU trial
MEASURED     = ["Lactate", "WBC", "Creatinine", "Platelets", "HCO3", "Glucose"]

np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| if 'cpu', set Runtime > Change runtime type > GPU for a faster LSTM")

## Step 3 — Quick EDA
Look at the three things that drive every later choice: **how rare sepsis is** (class imbalance), **how long patients stay** (sequence length for the LSTM), and **which columns are mostly missing** (what we'll impute).

In [ ]:
# EDA on a random 5,000-patient sample (reproducible)
rng = np.random.default_rng(SEED)
sample = sorted(rng.choice(files, size=min(5000, len(files)), replace=False))
n_pat = len(sample); n_septic = total_hours = pos_hours = 0; icu_lens = []; na = None
for f in sample:
    d = pd.read_csv(f, sep="|")
    if na is None: na = pd.Series(0.0, index=d.columns)
    lab = d["SepsisLabel"].values
    n_septic += int(lab.max() > 0); total_hours += len(d); pos_hours += int(lab.sum())
    icu_lens.append(len(d)); na += d.isna().sum()
print(f"patients sampled:      {n_pat}")
print(f"septic patients:       {n_septic} ({100*n_septic/n_pat:.1f}%)")
print(f"sepsis-positive hours: {pos_hours} ({100*pos_hours/total_hours:.2f}% of all hours)")
print(f"median ICU hours/pt:   {int(np.median(icu_lens))}")
print("\nMost-missing columns (fraction of hours missing):")
print((na / total_hours).sort_values(ascending=False).head(12).round(3))

## Step 4 — Load all patients & preprocess
Four standard ICU-time-series steps:
1. **Missingness flags** — for a few key labs, record *whether* it was measured. In the ICU, the fact a test was ordered is itself a signal.
2. **Forward-fill within each patient** — carry the last known value forward (a lab stays "current" until re-measured).
3. **Patient-level split** — split by *patient*, never by row, so no patient's hours leak across train/val/test.
4. **Impute + standardize using training stats only** — fill leftover gaps with train medians and z-score with train mean/std. Computing these on train only prevents leakage.

In [ ]:
# Load every patient into one DataFrame (adds a patient id). ~3-5 min for the full set.
from tqdm.auto import tqdm
use = files if MAX_PATIENTS is None else files[:MAX_PATIENTS]
frames = []
for pid, f in enumerate(tqdm(use, desc="loading")):
    d = pd.read_csv(f, sep="|"); d["pid"] = pid; frames.append(d)
df = pd.concat(frames, ignore_index=True).sort_values(["pid", "ICULOS"]).reset_index(drop=True)
LABEL = "SepsisLabel"
feat_cols = [c for c in df.columns if c not in (LABEL, "pid")]
print("combined shape:", df.shape)

In [ ]:
# 1) informative-missingness flags (computed BEFORE filling)
for c in MEASURED:
    df[f"{c}_measured"] = df[c].notna().astype(float)
ind_cols = [f"{c}_measured" for c in MEASURED]

# 2) forward-fill each column within a patient
df[feat_cols] = df.groupby("pid")[feat_cols].ffill()

# 3) patient-level split (a patient is never in two splits -> no leakage)
pids = df["pid"].unique(); rng = np.random.default_rng(SEED); rng.shuffle(pids)
n = len(pids)
tr_ids = set(pids[:int(.70*n)]); va_ids = set(pids[int(.70*n):int(.85*n)]); te_ids = set(pids[int(.85*n):])
split = df["pid"].map(lambda p: "tr" if p in tr_ids else ("va" if p in va_ids else "te"))

# 4) fill remaining NaN with TRAIN medians, then standardize with TRAIN mean/std
medians = df.loc[split == "tr", feat_cols].median(numeric_only=True)
df[feat_cols] = df[feat_cols].fillna(medians)
mean = df.loc[split == "tr", feat_cols].mean()
std  = df.loc[split == "tr", feat_cols].std().replace(0, 1.0)
df[feat_cols] = (df[feat_cols] - mean) / std

ALLFEAT = feat_cols + ind_cols
print("features:", len(ALLFEAT), "| positive-hour rate:", f"{100*df[LABEL].mean():.2f}%")

## Step 5 — Baseline: gradient-boosted trees
Treat each patient-hour as an independent row (no memory of earlier hours). Fast, strong, and the bar the LSTM must clear. We weight the rare positive class so the model doesn't just predict "no sepsis" everywhere. **Read AUPRC first** — under heavy imbalance AUROC looks flatteringly high.

In [ ]:
Xtr, ytr = df.loc[split == "tr", ALLFEAT].values, df.loc[split == "tr", LABEL].values
Xte, yte = df.loc[split == "te", ALLFEAT].values, df.loc[split == "te", LABEL].values

pos_w = (ytr == 0).sum() / max((ytr == 1).sum(), 1)   # how rare positives are
sw = np.where(ytr == 1, pos_w, 1.0)                    # balanced sample weights

clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_depth=6, random_state=SEED)
clf.fit(Xtr, ytr, sample_weight=sw)
p_base = clf.predict_proba(Xte)[:, 1]
base_auroc = roc_auc_score(yte, p_base); base_auprc = average_precision_score(yte, p_base)
print(f"[baseline] AUROC={base_auroc:.3f}  AUPRC={base_auprc:.3f}")

## Step 6 — LSTM: sequence model
Now give the model memory: each patient becomes a sequence of hourly feature vectors, and the LSTM predicts sepsis at **every** hour. Sequences have different lengths, so we pad each batch to its longest patient and **mask** the padding out of the loss. `pos_weight` handles the imbalance. We cap each patient to their last `MAX_LEN` hours to bound memory (a documented simplification — the positive labels sit near the end of a septic stay, so they're preserved).

In [ ]:
def build_sequences(ids):
    seqs = []; g = df[df["pid"].isin(ids)]
    for pid, grp in g.groupby("pid"):
        x = grp[ALLFEAT].values[-MAX_LEN:]; y = grp[LABEL].values[-MAX_LEN:]
        seqs.append((torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)))
    return seqs

class SeqDS(Dataset):
    def __init__(self, seqs): self.seqs = seqs
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i]

def collate(batch):
    xs, ys = zip(*batch)
    lens = torch.tensor([len(x) for x in xs])
    Xp = pad_sequence(xs, batch_first=True)                       # (B, Tmax, F)
    Yp = pad_sequence(ys, batch_first=True)                       # (B, Tmax)
    mask = (torch.arange(Xp.size(1))[None, :] < lens[:, None]).float()
    return Xp, Yp, lens, mask

class LSTMNet(nn.Module):
    def __init__(self, n_feat, hidden):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x, lens):
        packed = pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)        # (B, Tmax, H)
        return self.head(out).squeeze(-1)                          # (B, Tmax) logits

tr_dl = DataLoader(SeqDS(build_sequences(tr_ids)), batch_size=BATCH, shuffle=True,  collate_fn=collate)
te_dl = DataLoader(SeqDS(build_sequences(te_ids)), batch_size=BATCH, shuffle=False, collate_fn=collate)

model = LSTMNet(len(ALLFEAT), HIDDEN).to(DEV)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossfn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], dtype=torch.float32, device=DEV), reduction="none")

for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for Xp, Yp, lens, mask in tr_dl:
        Xp, Yp, mask = Xp.to(DEV), Yp.to(DEV), mask.to(DEV)
        logits = model(Xp, lens)
        loss = (lossfn(logits, Yp) * mask).sum() / mask.sum()      # loss only on real (non-pad) hours
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    print(f"epoch {ep+1}/{EPOCHS}  loss={tot/len(tr_dl):.4f}")

model.eval(); P, T = [], []
with torch.no_grad():
    for Xp, Yp, lens, mask in te_dl:
        p = torch.sigmoid(model(Xp.to(DEV), lens)).cpu().numpy()
        m = mask.numpy().astype(bool)
        P.append(p[m]); T.append(Yp.numpy()[m])
P = np.concatenate(P); T = np.concatenate(T)
lstm_auroc = roc_auc_score(T, P); lstm_auprc = average_precision_score(T, P)
print(f"[LSTM] AUROC={lstm_auroc:.3f}  AUPRC={lstm_auprc:.3f}  (device={DEV})")

## Step 7 — Compare & save
Side-by-side AUROC/AUPRC, plus a saved figure and CSV. If the LSTM doesn't clearly beat the baseline yet, that's a **real and common finding** — sequence models need tuning, and reporting it honestly is exactly the kind of analysis that made Project 1 strong.

In [ ]:
res = pd.DataFrame({"model": ["baseline (GBDT)", "LSTM"],
                    "AUROC": [base_auroc, lstm_auroc],
                    "AUPRC": [base_auprc, lstm_auprc]})
print(res.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2); w = 0.35
ax.bar(x - w/2, res["AUROC"], w, label="AUROC")
ax.bar(x + w/2, res["AUPRC"], w, label="AUPRC")
ax.set_xticks(x); ax.set_xticklabels(res["model"]); ax.set_ylim(0, 1)
ax.set_title("Sepsis early prediction — baseline vs LSTM"); ax.legend()
for i, (a, b) in enumerate(zip(res["AUROC"], res["AUPRC"])):
    ax.text(i - w/2, a + .01, f"{a:.2f}", ha="center")
    ax.text(i + w/2, b + .01, f"{b:.2f}", ha="center")
plt.tight_layout(); plt.savefig("/content/sepsis_results.png", dpi=130); plt.show()
res.to_csv("/content/sepsis_results.csv", index=False)
print("saved: /content/sepsis_results.png , /content/sepsis_results.csv")

## Step 8 — Lead-time analysis: how early does it warn?
For each septic patient, how many hours **before clinical onset** the LSTM first alarms — swept across three thresholds. This is the clinically meaningful early-warning number, shown against its alarm-fatigue cost (false alarms on non-septic patients). Onset is computed as first-positive-label-hour + 6, since PhysioNet's label flips on 6 h before actual onset.

In [ ]:
@torch.no_grad()
def patient_probs(pid):
    g = df[df["pid"] == pid]
    x = torch.tensor(g[ALLFEAT].values[-MAX_LEN:], dtype=torch.float32).unsqueeze(0).to(DEV)
    lens = torch.tensor([x.shape[1]])
    p = torch.sigmoid(model(x, lens)).cpu().numpy()[0]
    return p, g[LABEL].values[-MAX_LEN:]

model.eval()
test_pat  = [(pid, *patient_probs(pid)) for pid in te_ids]
septic    = [(pid, p, y) for pid, p, y in test_pat if y.max() > 0]
nonseptic = [(pid, p, y) for pid, p, y in test_pat if y.max() == 0]
print(f"test set: {len(septic)} septic, {len(nonseptic)} non-septic patients\n")

allp = np.concatenate([p for _, p, _ in test_pat])
print(f"{'alarm on':<14}{'caught':>9}{'median lead':>14}{'false alarms':>14}")
for pct in [90, 95, 99]:
    thr = np.percentile(allp, pct)
    leads = []; caught = 0
    for pid, p, y in septic:
        i0 = int(np.argmax(y > 0))       # first positive-label hour (= clinical onset - 6h)
        onset = i0 + 6
        alarm = int(np.argmax(p >= thr)) if (p >= thr).any() else None
        if alarm is not None and alarm <= onset:
            caught += 1; leads.append(onset - alarm)
    fa  = np.mean([(p >= thr).any() for _, p, _ in nonseptic])
    med = int(np.median(leads)) if leads else 0
    print(f"top {100-pct:>2}% hours {100*caught/len(septic):>7.0f}% {med:>11} h {100*fa:>11.0f}%")

## What you built & where to go next
A complete, reproducible sepsis pipeline with an honest baseline-vs-LSTM comparison.

**Next steps (Day 2+):**
- **Lead time** — how many hours *before* onset does the model fire? The clinically meaningful number.
- **Calibration** — are the predicted probabilities trustworthy? (reliability curve)
- **Failure analysis** — who does it miss, and when? (mirror Project 1's failure-analysis strength)
- **Tuning** — hidden size, epochs, GRU vs LSTM, class weighting; add the challenge's official utility score.
- **Alarm fatigue** — pick a threshold and report false alarms per true catch.

Paste your results back and we'll pick the next step together.